# Notebook 03 — Privacy, Governance & Compliance Audit
## NovaCred Credit Application Dataset
### Role: Data Governance Officer

---

This notebook is the formal governance and compliance audit for NovaCred's credit application pipeline. It is structured around four responsibilities of the Governance Officer role:

1. **PII Identification & Classification** — cataloguing every personally identifiable field and its risk level
2. **Pseudonymization Demonstration** — implementing and critically evaluating privacy-preserving techniques in code
3. **GDPR Compliance Mapping** — mapping observed pipeline gaps to specific regulatory obligations
4. **EU AI Act Classification & Bias Governance** — classifying the system under the EU AI Act
5. **Governance Controls & Recommendations** - proposing actionable controls
6. **Executive Summary** - summarizing key insights and requiredf actions


This audit builds on the findings of `01-data-quality.ipynb` (data engineering) and `02-data-analysis.ipynb` (bias detection). It does not repeat those analyses but interprets them through a regulatory and governance lens.

> **Regulatory Scope:** GDPR (EU) 2016/679 · EU AI Act (EU) 2024/1689

## Setup

In [ ]:
import pandas as pd
import numpy as np
import hashlib
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load the cleaned dataset produced by 01-data-quality.ipynb
df = pd.read_csv('clean_credit_applications.csv')

print(f"Dataset loaded: {df.shape[0]} records, {df.shape[1]} columns")

Dataset loaded: 489 records, 33 columns


---
## Part 1 — PII Identification & Classification

### Regulatory Foundation

The GDPR (Article 4(1)) defines personal data as any information relating to an identified or identifiable natural person. This definition is intentionally broad and creates two distinct risk tiers that determine how data must be treated throughout its lifecycle.

**Direct Identifiers** are fields that identify an individual on their own, without any additional information. Their exposure immediately and unambiguously ties back to a real person.

**Quasi-Identifiers** are fields that appear innocuous in isolation but become identifying when combined. A landmark study by Sweeney (2000) demonstrated that ZIP code, gender, and date of birth in combination are sufficient to uniquely identify approximately 87% of the US population — all three of these fields are present in this dataset.

The distinction matters operationally: direct identifiers must be pseudonymised or dropped for any analytical use; quasi-identifiers require generalisation techniques such as age banding or ZIP truncation to reduce re-identification risk.

---
## Part 2 — Pseudonymization Demonstration

### Regulatory Foundation

GDPR Article 4(5) defines pseudonymisation as processing personal data so it can no longer be attributed to a specific data subject without additional information kept separately. Three techniques exist — the choice depends on whether the original value needs to be recoverable for authorised purposes (e.g., fulfilling a GDPR Art. 17 erasure request):

| Technique | Reversible? | Still Personal Data? | Key Weakness |
|-----------|-------------|---------------------|--------------|
| **Plain SHA-256 hashing** | No | Yes | **Deterministic** — same input always yields same output. An attacker who knows the input space (e.g. all valid SSN formats) can pre-compute every possible hash and match against the dataset. No lookup table required for the attack. |
| **Salted SHA-256 hashing** | No (without salt) | Yes | Salt must be stored securely and separately. If the salt is exposed alongside the data, the protection collapses. Appropriate when recovery is not operationally needed. |
| **Tokenization** | Yes (via lookup table) | Yes | Lookup table must be secured; if stolen, all tokens are reversed. Enables GDPR Art. 17 erasure: delete the token from the table and the record becomes effectively anonymous. |

**Decision applied in this notebook:**
- **SSN, email, IP address** → Salted SHA-256. These fields have no business reason to be recovered after pseudonymisation. The salt makes pre-computation attacks infeasible.
- **Full name** → Tokenization. May need to be recovered by a compliance officer responding to a GDPR Art. 15 access or Art. 17 erasure request. A secured lookup table enables this without exposing the name in the analytical dataset.

---
## Part 3 — GDPR Compliance Mapping

### Regulatory Foundation

The GDPR establishes seven data protection principles in Article 5(1) that govern every stage of data processing. Compliance must be demonstrable at every stage of the data lifecycle — from collection through to deletion. Article 5(2) places the burden of proof on the data controller: NovaCred must be able to prove compliance, not merely claim it.

The audit below maps each relevant GDPR principle and article to the specific gaps identified in the NovaCred dataset and pipeline.

---
## Part 4 — EU AI Act Classification & Governance Controls

### Regulatory Foundation

The EU AI Act (Regulation 2024/1689), which entered into force in August 2024, establishes a risk-based classification framework for AI systems. **Credit scoring and creditworthiness assessment** is explicitly enumerated as a **High-Risk AI system** in Annex III, Point 5(b). This classification applies directly to NovaCred's automated loan approval system.

High-risk AI systems are subject to mandatory obligations under Chapter III, Section 2, including:

| Article | Obligation |
|---------|-----------|
| Art. 9  | A risk management system must be established and maintained throughout the lifecycle |
| Art. 10 | Training, validation, and testing data must meet quality criteria including examination for biases |
| Art. 13 | The system must be sufficiently transparent to enable the deployer to interpret its output |
| Art. 14 | The system must allow effective human oversight, including the ability to override outputs |
| Art. 15 | The system must maintain adequate accuracy, robustness, and cybersecurity with documented metrics |
| Art. 43 | The system must undergo a conformity assessment before deployment |

---
## Governance Controls & Recommendations



---
## Executive Summary
